Two workflows: one-shot & persistent Job

In [ ]:
import pyquick

WATER = """
    O  -0.33840   0.00380   0.23923
    H  -0.33510  -0.00190  -0.83277
    H   0.67350  -0.00190   0.59353
"""


## Workflow one-shot

Each call is completely self-contained: it sets up QUICK, runs, harvests the results, and
finalizes. Nothing is carried between calls.

In [2]:
calc = pyquick.PyQuick(method='HF', basis='STO-3G')

r = calc.get_energy(WATER)
print('energy      :', r.total_energy)

r = calc.get_grad(WATER)
print('gradient    :', r.gradient.shape)

r = calc.geo_opt(WATER)
print('optimized?  :', r.converged)

energy      : -74.94786381401461
gradient    : (3, 3)
optimized?  : True


## Workflow persistent `Job`
The atoms are fixed when the job is created. Only their coordinates change between runs.

1. sets up the molecule and settings **once** (`calc.new_job`, like AMBER API's `setQuickJob`),
2. runs each geometry with `job.run(geom)`, **carrying the converged density forward** ,
3. frees the memory with `job.delete()` (like `deleteQuickJob`).

In [3]:
# Four geometries of the same water molecule.
def stretch(geom, factor):
    """Scale every bond outward from the first atom — a crude symmetric stretch,
    used here just to make a series of geometries of the SAME molecule."""
    lines = [ln.split() for ln in geom.strip().splitlines()]
    ref = [float(v) for v in lines[0][1:4]]
    out = []
    for sym, *xyz in lines:
        p = [float(v) for v in xyz]
        p = [ref[i] + factor * (p[i] - ref[i]) for i in range(3)]
        out.append(f"{sym}  {p[0]:.6f}  {p[1]:.6f}  {p[2]:.6f}")
    return "\n".join(out)
    
scan = [stretch(WATER, f) for f in (1.00, 1.03, 1.06, 1.09)]

calc = pyquick.PyQuick(method='HF', basis='STO-3G')

job = calc.new_job(scan[0])         # allocate once; fix the molecule (3 atoms)

energies = []
for geom in scan:
    r = job.run(geom)        
    energies.append(r.total_energy)
    print(f'  E = {r.total_energy:.9f} ')

job.delete()                        # release the allocated setup
print('\nscan energies:', [round(e, 6) for e in energies])

  E = -74.947863814 
  E = -74.936703818 
  E = -74.923251866 
  E = -74.907925822 

scan energies: [-74.947864, -74.936704, -74.923252, -74.907926]
